In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 09, Foundations

Companion notebook to [09_foundations.md](09_foundations.md). Axiom vs theorem classification, efficient vs foundational mode, and `d² = 0` derived from the generator-level axiom.

## Default engine, every rule is an axiom

In [ ]:
from jacopy.proof.expansion import default_engine

eng = default_engine()
for d in eng.definitions:
    label = 'theorem' if d.is_theorem else 'axiom'
    print(f'{label:<8} | {d.name}')

## `d_squared_mode="theorem"`, reclassify

In [ ]:
eng_th = default_engine(d_squared_mode='theorem')
for d in eng_th.definitions:
    if d.name == 'd² = 0':
        print('is_theorem    :', d.is_theorem)
        print('has builder?  :', d.theorem_proof_builder() is not None)

## Efficient vs foundational, same step, different sub-proof

Efficient mode carries no children; foundational mode attaches a generator-level citation when a theorem-class rule fires.

In [ ]:
from jacopy.calculus.invariant_d import default_d
from jacopy.core.registry import PropertyRegistry
from jacopy.core.expr import Symbol, Integer
from jacopy.core.properties import Graded
from jacopy.proof.verifier import prove_equivalence

reg = PropertyRegistry()
omega = Symbol('ω')
reg.declare(omega, Graded(degree=2))
expr = default_d(default_d(omega))

eng_eff = default_engine(registry=reg, mode='efficient',
                         d_squared_mode='theorem')
eng_fnd = default_engine(registry=reg, mode='foundational',
                         d_squared_mode='theorem')

eff = prove_equivalence(expr, Integer(0), registry=reg, engine=eng_eff)
fnd = prove_equivalence(expr, Integer(0), registry=reg, engine=eng_fnd)

print('efficient steps:')
for s in eff.steps:
    print(f'  {s.rule:<15} children={len(s.children)}')
print('foundational steps:')
for s in fnd.steps:
    print(f'  {s.rule:<15} children={len(s.children)}')
    for c in s.children:
        print(f'     ↳ {c.rule}')

## What does the sub-proof say?

The foundational sub-proof's only input is `d(df) = 0`, the generic axiom the package treats as primitive. `d² = 0` extends from it at every form degree.

In [ ]:
child = fnd.steps[0].children[0]
print('child rule         :', child.rule)
print('child justification:\n  ', child.justification)

## Custom Definition, axiom class

`ExpansionEngine([YourDef()])` lets you assemble your own axiom set and run the engine on it. Below: a single rule that drives a `c_zero` symbol to zero.

In [ ]:
from jacopy.proof.expansion import Definition, ExpansionEngine
from jacopy.core.expr import Sum

class ZeroConstAxiom(Definition):
    name = 'c_zero := 0 (axiom)'

    def matches(self, expr):
        return isinstance(expr, Symbol) and expr.name == 'c_zero'

    def rewrite(self, expr):
        return Integer(0)

custom_engine = ExpansionEngine([ZeroConstAxiom()])
c, x = Symbol('c_zero'), Symbol('x')
expanded, steps = custom_engine.expand(Sum(c, x))
print('expanded:', expanded)
print('rule   :', steps[0].rule)
print('prov   :', steps[0].provenance_tag)

## Custom Definition, theorem class

Override `theorem_proof_builder` and the same rule becomes a theorem; foundational mode attaches the sub-proof.

In [ ]:
from jacopy.proof.chain import ProofChain
from jacopy.proof.step import ProofStep

class ZeroConstTheorem(Definition):
    name = 'c_zero := 0 (theorem)'

    def matches(self, expr):
        return isinstance(expr, Symbol) and expr.name == 'c_zero'

    def rewrite(self, expr):
        return Integer(0)

    def theorem_proof_builder(self):
        def build(matched):
            return ProofChain(steps=[
                ProofStep(
                    rule='c_zero = c_zero − c_zero (axiom)',
                    before=matched, after=Integer(0),
                    justification='self-annihilation axiom',
                    provenance_tag='axiom',
                ),
            ])
        return build

eff = ExpansionEngine([ZeroConstTheorem()], mode='efficient')
fnd = ExpansionEngine([ZeroConstTheorem()], mode='foundational')

c = Symbol('c_zero')
_, eff_steps = eff.expand(c)
_, fnd_steps = fnd.expand(c)

print('efficient children :', len(eff_steps[0].children))
print('foundational child :', len(fnd_steps[0].children))
print('  ↳ sub-rule       :', fnd_steps[0].children[0].rule)

## Theorem Book structure

The `Theorem` dataclass carries five fields: `name`, `statement`, `from_axioms`, `proof`, `notes`. The singleton `theorem_book` registry holds seeded theorems; downstream code pulls `proof` and embeds the chain inside a larger proof.

In [ ]:
from jacopy.library.theorem_book import Theorem
from jacopy.library import theorem_book
import dataclasses

print('fields:', [f.name for f in dataclasses.fields(Theorem)])
print('registry size:', len(theorem_book))
print()
for name in theorem_book.names():
    t = theorem_book.get(name)
    print(f'{name:32s} proof_len={len(t.proof)}')

## Three provenance layers

Property (symbol), Definition (expansion), Theorem, all three carry the `axiom`/`theorem` distinction. `Theorem.from_axioms` lands a single citation inside a paper-style proof.

In [ ]:
for name in ('poisson_jacobi', 'courant_jacobi_twist'):
    thm = theorem_book.get(name)
    print(name, '->', thm.from_axioms)

## End of the tutorial series

Nine chapters complete. The package is now a tool ready to be extended with your own brackets, your own theorems, and your own axiom sets.